# 04 - Is it any good? 📏

In the workshop, a lower loss after fine-tuning *didn't* mean a better model.

Same here: an answer that sounds good tells you little. We measure three layers,
cheapest first:

1. **Retrieval**: are the right films in the top 5?
2. **Parsing**: are the filters extracted correctly?
3. **Generation**: is the answer grounded in what was retrieved?

The questions are in `eval/questions.json`. ⚠️ I drafted the answer keys from your list:
**check them**, since an eval set is only as good as its labels.

In [1]:
# Make the project importable even if it wasn't installed (uv sync installs it)
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

%load_ext autoreload
%autoreload 2

from movierag.config import get_config
cfg = get_config()
print(cfg.summary())

bookmarks     : /home/jp/llms/rag/movies-reels.json
state dir     : /home/jp/.movierag
LLM           : hf:Qwen/Qwen3-4B-Instruct-2507 (4-bit)
embeddings    : BAAI/bge-small-en-v1.5
reranker      : cross-encoder/ms-marco-MiniLM-L6-v2
TMDb key      : yes
YouTube key   : MISSING
watch region  : CH
offline mode  : False


In [2]:
import pandas as pd
from movierag.app import load_components
from movierag.evaluate import load_eval_set

data = load_eval_set(cfg.bookmarks_path.parent / "eval" / "questions.json")
print(len(data["retrieval"]), "retrieval questions,", len(data["parsing"]), "parsing questions")

c = load_components(with_llm=False)   # retrieval doesn't need the LLM: keep the GPU free

30 retrieval questions, 12 parsing questions


/home/jp/llms/rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1783.22it/s]


## 1. Retrieval: an ablation study 🔬

Same questions, four retrieval setups. Metrics:

- **hit@5**: is at least one right film in the top 5?
- **recall@5**: what share of the right films are in the top 5?
- **MRR**: 1 if the first result is right, ½ if the second is, ⅓...

In [3]:
from movierag.evaluate import evaluate_retrieval

def searcher(method, reranker=None):
    def search(q):
        hits = c.index.search(q, method=method, top_k=5, reranker=reranker)
        return [c.index.by_id[h.node.node_id].imdb_id for h in hits]
    return search

setups = {
    "vector": searcher("vector"),
    "bm25": searcher("bm25"),
    "hybrid": searcher("hybrid"),
    "hybrid + rerank": searcher("hybrid", c.reranker),
}
results, details = {}, {}
for name, fn in setups.items():
    results[name], details[name] = evaluate_retrieval(data["retrieval"], fn, k=5)

pd.DataFrame(results).T.round(3)

,hit@5,recall@5,mrr
vector,1.0,0.972,0.918
bm25,1.0,0.961,0.922
hybrid,1.0,0.989,0.967
hybrid + rerank,1.0,0.989,0.983


### Where does it fail? 🔍

Averages hide the interesting part. Look at the misses one by one: is the label wrong,
the document poor (no plot?), or the retriever weak?

In [4]:
best = "hybrid + rerank"
by_imdb = {t.imdb_id: t.title for t in c.titles}
for row, q in zip(details[best], data["retrieval"]):
    if row["mrr"] < 1:
        print(f"MRR {row['mrr']:.2f} | {row['question']}")
        print("    wanted:", [by_imdb.get(i, i) for i in q["relevant_imdb"]])
        print("    got   :", [by_imdb.get(i, i) for i in row["top"]])

MRR 0.50 | a clone sent on dangerous missions who keeps dying and being reprinted
    wanted: ['Mickey 17']
    got   : ['The Tank', 'Mickey 17', 'Die Another Day', 'Flight of Fury', 'Alien Invasion: S.U.M.1']


## 2. Parsing: rules vs LLM 🧾

Now we need the LLM.

In [5]:
from movierag.models import load_llm
from movierag.query_parser import parse_query
from movierag.evaluate import plan_field_matches

llm = load_llm(cfg)
c.recommender.llm = llm

rows = []
for item in data["parsing"]:
    for mode, model in (("rules", None), ("llm+rules", llm)):
        plan, how = parse_query(item["question"], model)
        for field, ok in plan_field_matches(plan, item["expected"]).items():
            rows.append({"mode": mode, "field": field, "ok": ok, "question": item["question"]})

parse_df = pd.DataFrame(rows)
parse_df.pivot_table(index="field", columns="mode", values="ok", aggfunc="mean").round(2)

Loading weights: 100%|██████████| 398/398 [00:11<00:00, 35.49it/s]
LLM query parsing failed (1 validation error for QueryPlan
kinds
  Input should be a valid array [type=list_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/list_type); using rules only


mode,llm+rules,rules
field,,
genres,1.00,1.0
kinds,1.00,1.0
max_runtime_min,1.00,1.0
max_year,1.00,1.0
min_year,0.67,1.0
similar_to,1.00,1.0
wants_new,1.00,1.0


In [6]:
parse_df[~parse_df["ok"]].sort_values(["mode", "question"])

,mode,field,ok,question
20,llm+rules,min_year,False,an 80s or older kung fu movie


## 3. Generation: grounded or not? 🧷

Two checks:

- **Citations** (free, deterministic): every `[L1]`/`[N2]` in the answer must exist.
- **LLM-as-judge**: another LLM call grades whether the claims are supported by the
  candidates. ⚠️ Here the judge is the same small model being judged, which is weak.
  Use a stronger model as the judge (e.g. the Modal one) for numbers you trust.

In [7]:
import time
from movierag.evaluate import judge_faithfulness
from movierag.recommender import context_text

gen_rows = []
for q in data["end_to_end"]:
    r = c.recommender.recommend(q)
    mine, new = context_text(r)
    passed, verdict = judge_faithfulness(llm, r.answer, mine + "\n\n" + new)
    gen_rows.append({
        "question": q, "parsed_by": r.parsed_by, "n_mine": len(r.my_list), "n_new": len(r.new),
        "cited": len(r.cited), "bad_citations": len(r.bad_citations),
        "judge": "PASS" if passed else "FAIL", "seconds": round(r.timings["total"], 1),
    })
    print(f"--- {q}\n{r.answer}\n>> judge: {verdict}\n")

pd.DataFrame(gen_rows)

LLM query parsing failed (1 validation error for QueryPlan
kinds
  Input should be a valid array [type=list_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/list_type); using rules only


--- I have 90 minutes tonight, something gripping
- [L1] Bloodhounds (2023) – A gripping 61-minute drama-action series about two young boxers uniting to fight ruthless foes and protect their loved ones; high-stakes, action-driven, and emotionally intense, perfect for a thrilling 90-minute movie-night.  
- [L3] The Weapon (2012) – An 85-minute drama centered on military weapons testing and tense moral dilemmas; features high tension, real-world stakes, and a strong narrative arc involving danger and secrecy.  
- [L4] Tomb Adventurer (2022) – A full 90-minute action-fantasy film with mythical beasts, high-risk adventures, and a clear quest to escape a deadly labyrinth; packed with suspense and excitement, ideal for a gripping evening.  

*Note: [L2] Devs is 52 minutes per episode and not a single film, making it unsuitable for a 90-minute continuous watch. [L5] Subdue the Devil is 82 minutes but has lower ratings and less emphasis on sustained gripping tension.*
>> judge: PASS  
Each rec

,question,parsed_by,n_mine,n_new,cited,bad_citations,judge,seconds
0,"I have 90 minutes tonight, something gripping",rules,5,0,5,0,PASS,21.3
1,"a war movie, but not too long",llm,5,0,3,0,PASS,19.0
2,a documentary about the ocean,llm,0,3,3,0,PASS,21.8
3,something like Under the Skin,llm,5,0,2,0,PASS,17.4
4,new sci-fi series I haven't saved,llm,5,3,3,0,PASS,21.5
5,a youtube video essay about kung fu cinema und...,llm,0,0,0,0,PASS,10.4


# ✅ What we just measured

You now have numbers for each layer, so every change becomes an experiment:

| Try this | Expect it to move |
|---|---|
| `EMBED_MODEL=BAAI/bge-base-en-v1.5` or `BAAI/bge-m3` (multilingual) | retrieval metrics |
| `RERANK_MODEL=BAAI/bge-reranker-v2-m3` | MRR |
| Add more text to `Title.embedding_text()` | recall on vague questions |
| `LLM_BACKEND=openai_like` with the 30B model on Modal | parsing + judge pass rate (and latency, cost) |
| Ollama `qwen3.5:9b` vs `qwen3.5:4b` | parsing accuracy vs speed |

Change one thing, rerun this notebook, compare the tables. Same discipline as the
workshop's *"change one knob, ask what changed"*.